In [ ]:
# Offensive AI GGUF Export Kernel (Checkpoint-1875)
import os
with open('export_runner.py', 'w', encoding='utf-8') as f:
    f.write('''import os
import sys
import time
import json
import shutil
import hashlib
import subprocess
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(1048576):
            h.update(chunk)
    return h.hexdigest()

print("=" * 80)
print("STARTING GGUF EXPORT FOR OFFENSIVE AI (CHECKPOINT-1875)")
print("=" * 80)

WORK_DIR = Path("/kaggle/working")
MERGED_DIR = WORK_DIR / "qwen2.5-1.5b-offensive-ai-merged"
FP16_GGUF = WORK_DIR / "qwen2.5-1.5b-offensive-ai.f16.gguf"
Q4_GGUF = WORK_DIR / "qwen2.5-1.5b-offensive-ai.Q4_K_M.gguf"

# 1. Locate Adapter Directory
adapter_candidates = [
    Path("/kaggle/input/offensive-ai-final-adapter"),
    Path("/kaggle/input/offensive-ai-final-adapter/final_adapter"),
]
ADAPTER_DIR = None
for cand in adapter_candidates:
    if (cand / "adapter_model.safetensors").exists():
        ADAPTER_DIR = cand
        break

if not ADAPTER_DIR:
    # Search all inputs
    for p in Path("/kaggle/input").rglob("adapter_model.safetensors"):
        ADAPTER_DIR = p.parent
        break

assert ADAPTER_DIR and (ADAPTER_DIR / "adapter_model.safetensors").exists(), "Could not locate adapter_model.safetensors in /kaggle/input!"
print(f"Located Adapter Directory: {ADAPTER_DIR}")

# Verify adapter weights exist and compute hash
safetensors_path = ADAPTER_DIR / "adapter_model.safetensors"
adapter_hash = compute_sha256(safetensors_path)
print(f"Adapter Safetensors Path:   {safetensors_path}")
print(f"Adapter Safetensors Size:   {safetensors_path.stat().st_size:,} bytes")
print(f"Adapter Safetensors SHA256: {adapter_hash}")
assert adapter_hash == "39fd053a1725e15c9f8071578965d9b7d2b58b8aff2d0d722fc36d8796879327", "Hash mismatch with verified checkpoint-1875!"
print("VERIFIED: Source adapter matches checkpoint-1875 bit-for-bit!")

# 2. Install requirements & fix torchao conflict
print("\\nFixing torchao conflict and installing requirements...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gguf"], check=True)

# 3. Load Base Model in FP16
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"\\nLoading base model {BASE_MODEL_ID} in float16...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

# 4. Attach LoRA Adapter
print(f"\\nAttaching LoRA adapter from {ADAPTER_DIR}...")
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR), torch_dtype=torch.float16)

# 5. Merge and Unload
print("\\nMerging LoRA adapter weights into base model (merge_and_unload)...")
t0 = time.time()
merged_model = model.merge_and_unload()
print(f"Merge completed in {time.time() - t0:.2f}s")

# 6. Save Merged FP16 Model
print(f"\\nSaving merged model to {MERGED_DIR}...")
MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
tokenizer.save_pretrained(str(MERGED_DIR))
print("Merged model saved successfully.")

# Free python memory
del merged_model, model, base_model
import gc
gc.collect()

# 7. Clone llama.cpp and compile llama-quantize
print("\\nSetting up llama.cpp...")
LLAMA_CPP_DIR = WORK_DIR / "llama.cpp"
if not LLAMA_CPP_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggerganov/llama.cpp.git"], check=True, cwd=str(WORK_DIR))

print("Compiling llama-quantize via cmake...")
subprocess.run(["cmake", "-B", "build", "-DGGML_CUDA=OFF"], check=True, cwd=str(LLAMA_CPP_DIR))
subprocess.run(["cmake", "--build", "build", "--config", "Release", "-t", "llama-quantize", "-j", "4"], check=True, cwd=str(LLAMA_CPP_DIR))

QUANTIZE_BIN = LLAMA_CPP_DIR / "build" / "bin" / "llama-quantize"
if not QUANTIZE_BIN.exists():
    QUANTIZE_BIN = LLAMA_CPP_DIR / "build" / "llama-quantize"
assert QUANTIZE_BIN.exists(), f"llama-quantize binary not found at {QUANTIZE_BIN}"
print(f"Compiled llama-quantize at: {QUANTIZE_BIN}")

# 8. Convert HF to FP16 GGUF
print(f"\\nConverting merged model to FP16 GGUF: {FP16_GGUF}...")
CONVERT_SCRIPT = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"
subprocess.run([sys.executable, str(CONVERT_SCRIPT), str(MERGED_DIR), "--outfile", str(FP16_GGUF), "--outtype", "f16"], check=True)
fp16_size = FP16_GGUF.stat().st_size
fp16_hash = compute_sha256(FP16_GGUF)
print(f"FP16 GGUF generated: {fp16_size:,} bytes ({fp16_size/(1024**3):.2f} GB)")
print(f"FP16 GGUF SHA256: {fp16_hash}")

# 9. Quantize FP16 GGUF to Q4_K_M
print(f"\\nQuantizing FP16 GGUF to Q4_K_M: {Q4_GGUF}...")
subprocess.run([str(QUANTIZE_BIN), str(FP16_GGUF), str(Q4_GGUF), "Q4_K_M"], check=True)
q4_size = Q4_GGUF.stat().st_size
q4_hash = compute_sha256(Q4_GGUF)
print(f"Q4_K_M GGUF generated: {q4_size:,} bytes ({q4_size/(1024**2):.2f} MB)")
print(f"Q4_K_M GGUF SHA256: {q4_hash}")

# 10. Clean up intermediate merged weights and llama.cpp repo to conserve disk space
print("\\nCleaning up intermediate directory to save space...")
shutil.rmtree(MERGED_DIR, ignore_errors=True)
shutil.rmtree(LLAMA_CPP_DIR, ignore_errors=True)

# 11. Write Export Manifest
manifest = {
    "status": "SUCCESS",
    "base_model": BASE_MODEL_ID,
    "source_adapter": "checkpoint-1875 (final_adapter)",
    "adapter_sha256": adapter_hash,
    "exported_artifacts": {
        "fp16_gguf": {
            "filename": FP16_GGUF.name,
            "size_bytes": fp16_size,
            "size_gb": round(fp16_size / (1024**3), 2),
            "sha256": fp16_hash
        },
        "q4_k_m_gguf": {
            "filename": Q4_GGUF.name,
            "size_bytes": q4_size,
            "size_mb": round(q4_size / (1024**2), 2),
            "sha256": q4_hash
        }
    },
    "conversion_parameters": {
        "fp16_tool": "convert_hf_to_gguf.py",
        "quantization_tool": "llama-quantize",
        "quantization_type": "Q4_K_M"
    }
}
manifest_path = WORK_DIR / "export_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Wrote export manifest to: {manifest_path}")
print("\\n" + "=" * 80)
print("GGUF EXPORT COMPLETED SUCCESSFULLY!")
print("=" * 80)
''')
!python export_runner.py
